# Diabetes Data Cleaning Check

This notebook inspects `diabetes.csv` for data quality issues and builds a cleaned dataset ready for modeling.

Checks included:
- Shape, schema, missing values, duplicates
- Zero-as-missing values in medical columns
- Outlier detection (IQR method)
- Class balance

Cleaning actions included:
- Replace impossible zeros with `NaN` in key medical columns
- Impute with median values
- Optional outlier clipping using IQR bounds

In [ ]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

In [ ]:
df = pd.read_csv('diabetes.csv')
print('Shape:', df.shape)
print('\nDtypes:')
print(df.dtypes)
df.head()

In [ ]:
print('Duplicate rows:', int(df.duplicated().sum()))
print('\nMissing values by column:')
print(df.isna().sum())

In [ ]:
medical_zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

zero_report = pd.DataFrame({
    'zero_count': [(df[c] == 0).sum() for c in medical_zero_cols],
    'zero_percent': [((df[c] == 0).mean() * 100) for c in medical_zero_cols],
}, index=medical_zero_cols).sort_values('zero_count', ascending=False)

zero_report

In [ ]:
def iqr_outlier_report(dataframe):
    rows = []
    for col in dataframe.columns:
        q1 = dataframe[col].quantile(0.25)
        q3 = dataframe[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        n = int(((dataframe[col] < lower) | (dataframe[col] > upper)).sum())
        rows.append({
            'column': col,
            'outlier_count': n,
            'outlier_percent': round((n / len(dataframe)) * 100, 2),
            'lower_bound': round(lower, 3),
            'upper_bound': round(upper, 3),
        })
    return pd.DataFrame(rows).sort_values('outlier_count', ascending=False)

iqr_report = iqr_outlier_report(df)
iqr_report

In [ ]:
print('Class distribution (Outcome):')
print(df['Outcome'].value_counts().sort_index())
print('\nClass percentages:')
print((df['Outcome'].value_counts(normalize=True).sort_index() * 100).round(2))

## Cleaning Pipeline

Recommended for this dataset:
1. Replace impossible zeros with `NaN` in `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, `BMI`.
2. Impute those missing values with median.
3. Optional: clip outliers by IQR bounds.

In [ ]:
clean_df = df.copy()

# Step 1: replace impossible zeros with NaN
clean_df[medical_zero_cols] = clean_df[medical_zero_cols].replace(0, np.nan)

# Step 2: median imputation
median_values = clean_df[medical_zero_cols].median()
clean_df[medical_zero_cols] = clean_df[medical_zero_cols].fillna(median_values)

print('Missing values after median imputation:')
print(clean_df[medical_zero_cols].isna().sum())
clean_df.head()

In [ ]:
def clip_iqr(dataframe, cols):
    capped = dataframe.copy()
    for col in cols:
        q1 = capped[col].quantile(0.25)
        q3 = capped[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        capped[col] = capped[col].clip(lower=lower, upper=upper)
    return capped

numeric_cols = [c for c in clean_df.columns if c != 'Outcome']
clean_df_capped = clip_iqr(clean_df, numeric_cols)

print('Outlier report after optional clipping:')
iqr_outlier_report(clean_df_capped)

In [ ]:
# Save cleaned versions
clean_df.to_csv('diabetes_cleaned_median.csv', index=False)
clean_df_capped.to_csv('diabetes_cleaned_median_iqr_capped.csv', index=False)

print('Saved: diabetes_cleaned_median.csv')
print('Saved: diabetes_cleaned_median_iqr_capped.csv')

## Summary

From this dataset, common cleaning needs are:
- No duplicate rows and no explicit `NaN` values in the raw file
- But many impossible zeros in medical columns (especially `Insulin` and `SkinThickness`)
- Noticeable outliers in several numerical features

Use `diabetes_cleaned_median.csv` as a baseline cleaned file, and `diabetes_cleaned_median_iqr_capped.csv` if you want outlier-capped training data.

## Model Benchmarking (Classification)

The target column `Outcome` is binary (0/1), so this is a **classification** problem.
`LogisticRegression` is included because it is a regression-based classifier, and we also compare multiple tree, boosting, distance-based, and probabilistic models.

This section builds a **comparison platform** using:
- Stratified 5-fold cross-validation metrics
- Holdout test metrics
- Composite ranking score across metrics

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB

# Use the cleaned dataframe generated above.
# If this cell is run independently, rebuild clean_df.
if 'clean_df' not in globals():
    base_df = pd.read_csv('diabetes.csv').drop_duplicates()
    medical_zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
    base_df[medical_zero_cols] = base_df[medical_zero_cols].replace(0, np.nan)
    clean_df = base_df.fillna(base_df.mean(numeric_only=True))

X = clean_df.drop(columns=['Outcome'])
y = clean_df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    'LogisticRegression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=3000, random_state=42))
    ]),
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier(n_neighbors=9))
    ]),
    'SVC-RBF': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(kernel='rbf', probability=True, random_state=42))
    ]),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=400, random_state=42),
    'ExtraTrees': ExtraTreesClassifier(n_estimators=400, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(random_state=42),
    'AdaBoost': AdaBoostClassifier(random_state=42),
    'GaussianNB': GaussianNB(),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc',
}

rows = []
for name, model in models.items():
    cv_scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None

    rows.append({
        'model': name,
        'cv_accuracy_mean': cv_scores['test_accuracy'].mean(),
        'cv_precision_mean': cv_scores['test_precision'].mean(),
        'cv_recall_mean': cv_scores['test_recall'].mean(),
        'cv_f1_mean': cv_scores['test_f1'].mean(),
        'cv_roc_auc_mean': cv_scores['test_roc_auc'].mean(),
        'test_accuracy': accuracy_score(y_test, y_pred),
        'test_precision': precision_score(y_test, y_pred),
        'test_recall': recall_score(y_test, y_pred),
        'test_f1': f1_score(y_test, y_pred),
        'test_roc_auc': roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan,
    })

comparison_df = pd.DataFrame(rows)

# Composite ranking platform: average rank over key CV metrics.
rank_metrics = ['cv_accuracy_mean', 'cv_precision_mean', 'cv_recall_mean', 'cv_f1_mean', 'cv_roc_auc_mean']
for m in rank_metrics:
    comparison_df[f'rank_{m}'] = comparison_df[m].rank(ascending=False, method='min')
comparison_df['rank_avg'] = comparison_df[[f'rank_{m}' for m in rank_metrics]].mean(axis=1)

leaderboard = comparison_df.sort_values(['cv_roc_auc_mean', 'cv_f1_mean'], ascending=False).reset_index(drop=True)
ranked_platform = comparison_df.sort_values(['rank_avg', 'cv_roc_auc_mean'], ascending=[True, False]).reset_index(drop=True)

print('Leaderboard by CV ROC-AUC then CV F1:')
display(leaderboard[['model', 'cv_accuracy_mean', 'cv_precision_mean', 'cv_recall_mean', 'cv_f1_mean', 'cv_roc_auc_mean', 'test_accuracy', 'test_precision', 'test_recall', 'test_f1', 'test_roc_auc']].round(4))

print('Composite ranking platform (lower rank_avg is better):')
display(ranked_platform[['model', 'rank_avg', 'cv_accuracy_mean', 'cv_precision_mean', 'cv_recall_mean', 'cv_f1_mean', 'cv_roc_auc_mean']].round(4))

best_cv_auc = leaderboard.loc[0, 'model']
best_composite = ranked_platform.loc[0, 'model']

print(f'Best model by CV ROC-AUC: {best_cv_auc}')
print(f'Best model by composite rank: {best_composite}')

Leaderboard by CV ROC-AUC then CV F1:


,model,cv_accuracy_mean,cv_precision_mean,cv_recall_mean,cv_f1_mean,cv_roc_auc_mean,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
0,LogisticRegression,0.7866,0.7539,0.5842,0.6567,0.8436,0.6948,0.5778,0.4815,0.5253,0.8117
1,SVC-RBF,0.7769,0.7414,0.5749,0.6416,0.8319,0.7338,0.6444,0.5370,0.5859,0.7912
2,GaussianNB,0.7639,0.6906,0.6031,0.6406,0.8288,0.6948,0.5574,0.6296,0.5913,0.7696
3,ExtraTrees,0.7524,0.6778,0.5656,0.6141,0.8236,0.7273,0.6200,0.5741,0.5962,0.8192
4,AdaBoost,0.7655,0.7035,0.5703,0.6279,0.8175,0.7468,0.6596,0.5741,0.6139,0.8180
5,RandomForest,0.7606,0.6982,0.5656,0.6220,0.8156,0.7403,0.6591,0.5370,0.5918,0.8170
6,GradientBoosting,0.7443,0.6549,0.5794,0.6129,0.8156,0.7662,0.6957,0.5926,0.6400,0.8261
7,KNN,0.7410,0.6568,0.5701,0.6076,0.8069,0.7403,0.6346,0.6111,0.6226,0.7960
8,DecisionTree,0.6726,0.5314,0.5748,0.5495,0.6499,0.6688,0.5333,0.4444,0.4848,0.6172


Composite ranking platform (lower rank_avg is better):


,model,rank_avg,cv_accuracy_mean,cv_precision_mean,cv_recall_mean,cv_f1_mean,cv_roc_auc_mean
0,LogisticRegression,1.2,0.7866,0.7539,0.5842,0.6567,0.8436
1,SVC-RBF,2.4,0.7769,0.7414,0.5749,0.6416,0.8319
2,GaussianNB,3.2,0.7639,0.6906,0.6031,0.6406,0.8288
3,AdaBoost,4.2,0.7655,0.7035,0.5703,0.6279,0.8175
4,RandomForest,5.6,0.7606,0.6982,0.5656,0.6220,0.8156
5,ExtraTrees,6.0,0.7524,0.6778,0.5656,0.6141,0.8236
6,GradientBoosting,6.4,0.7443,0.6549,0.5794,0.6129,0.8156
7,KNN,7.6,0.7410,0.6568,0.5701,0.6076,0.8069
8,DecisionTree,8.2,0.6726,0.5314,0.5748,0.5495,0.6499


Best model by CV ROC-AUC: LogisticRegression
Best model by composite rank: LogisticRegression
